# M5 Walmart Demand Forecasting — Phase 2: Model Training & Comparison

**Models:** SARIMA · Prophet · LSTM  
**Evaluation metrics:** RMSE · MAE · MAPE · SMAPE  
**Experiment tracking:** MLflow  


---
## 1. Setup & Data Loading


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import warnings
warnings.filterwarnings("ignore")

from src.data_loader import load_ca1_daily
from src.evaluation import evaluate_all
from src.visualizations import plot_forecast, plot_residuals, plot_metrics_comparison

df = load_ca1_daily()
print(df.shape)
df.head()

---
## 2. Train / Test Split


In [ ]:
TEST_DAYS = 28  # forecast horizon

train = df.iloc[:-TEST_DAYS].copy()
test  = df.iloc[-TEST_DAYS:].copy()

print(f"Train: {len(train)} days  ({train['date'].min().date()} → {train['date'].max().date()})")
print(f"Test : {len(test)} days  ({test['date'].min().date()} → {test['date'].max().date()})")

y_test = test["sales"].values

---
## 3. SARIMA


In [ ]:
from src.models.sarima_model import train_sarima, predict_sarima

# Start MLflow run
mlflow.set_experiment("m5-ca1-forecasting")

with mlflow.start_run(run_name="SARIMA"):
    sarima_order = (1, 1, 1)
    sarima_seasonal = (1, 1, 1, 7)

    sarima_fit = train_sarima(train["sales"], order=sarima_order, seasonal_order=sarima_seasonal)
    sarima_preds = predict_sarima(sarima_fit, steps=TEST_DAYS)

    sarima_metrics = evaluate_all(y_test, sarima_preds, model_name="SARIMA")
    mlflow.log_params({"order": str(sarima_order), "seasonal_order": str(sarima_seasonal)})
    mlflow.log_metrics({k: float(v) for k, v in sarima_metrics.drop(columns="model").iloc[0].items()})

print(sarima_metrics)

In [ ]:
fig = plot_forecast(train, test, pd.Series(sarima_preds), model_name="SARIMA")
fig.savefig("../outputs/plots/sarima_forecast.png", dpi=150)
plt.show()

---
## 4. Prophet


In [ ]:
from src.models.prophet_model import build_prophet_df, train_prophet, predict_prophet, extract_forecast_values

with mlflow.start_run(run_name="Prophet"):
    train_prophet_df = build_prophet_df(train)

    prophet_model = train_prophet(train_prophet_df)
    prophet_forecast = predict_prophet(prophet_model, periods=TEST_DAYS)
    prophet_preds = extract_forecast_values(prophet_forecast, n_test=TEST_DAYS)

    prophet_metrics = evaluate_all(y_test, prophet_preds, model_name="Prophet")
    mlflow.log_metrics({k: float(v) for k, v in prophet_metrics.drop(columns="model").iloc[0].items()})

print(prophet_metrics)

In [ ]:
fig = plot_forecast(train, test, pd.Series(prophet_preds), model_name="Prophet")
fig.savefig("../outputs/plots/prophet_forecast.png", dpi=150)
plt.show()

---
## 5. LSTM


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from src.models.lstm_model import create_sequences, build_lstm_model, train_lstm, predict_lstm

LOOK_BACK = 28

scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train[["sales"]]).flatten()
full_scaled  = scaler.transform(df[["sales"]]).flatten()

# Build sequences from training data only
X_train_seq, y_train_seq = create_sequences(train_scaled, look_back=LOOK_BACK)

# Build test sequences from the overlap window (last LOOK_BACK train points + test)
test_window = full_scaled[-(TEST_DAYS + LOOK_BACK):]
X_test_seq, _ = create_sequences(test_window, look_back=LOOK_BACK)

with mlflow.start_run(run_name="LSTM"):
    lstm_model = build_lstm_model(look_back=LOOK_BACK, units=64)
    history = train_lstm(lstm_model, X_train_seq, y_train_seq, epochs=50, batch_size=32)
    lstm_preds = predict_lstm(lstm_model, X_test_seq, scaler=scaler)

    lstm_metrics = evaluate_all(y_test, lstm_preds, model_name="LSTM")
    mlflow.log_params({"look_back": LOOK_BACK, "units": 64})
    mlflow.log_metrics({k: float(v) for k, v in lstm_metrics.drop(columns="model").iloc[0].items()})

print(lstm_metrics)

In [ ]:
fig = plot_forecast(train, test, pd.Series(lstm_preds), model_name="LSTM")
fig.savefig("../outputs/plots/lstm_forecast.png", dpi=150)
plt.show()

---
## 6. Model Comparison


In [ ]:
all_metrics = pd.concat([sarima_metrics, prophet_metrics, lstm_metrics], ignore_index=True)
all_metrics = all_metrics.set_index("model").round(2)

# Save to disk
all_metrics.to_csv("../outputs/metrics/model_comparison.csv")
print(all_metrics)

In [ ]:
fig = plot_metrics_comparison(all_metrics.reset_index())
fig.savefig("../outputs/plots/model_comparison.png", dpi=150)
plt.show()

---
## 7. Results Summary

| Model | RMSE | MAE | MAPE | SMAPE |
|---|---|---|---|---|
| SARIMA | — | — | — | — |
| Prophet | — | — | — | — |
| LSTM | — | — | — | — |

*(Fill in after training)*

**Winner:** *(to be determined)*

**Next step:** → `notebooks/03_business_impact.ipynb` — translate forecast accuracy into inventory cost savings.
